# MRI fine-tuning, calibration and run status

This notebook inspects the versioned improvement experiment without starting training.
MRNet supplies training labels. KneeMRI and fastMRI remain external reference evaluations.
Study separation does not prove patient separation. Previous evaluation cohorts are not new untouched tests.
The 100 reserved calibration cases were removed from the candidate fitting set; the incumbent cannot be calibrated on them because it previously trained on them.

In [1]:
from pathlib import Path
import json
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'config.yaml').exists(): ROOT=ROOT.parent
run=ROOT/'runs/mri_finetune_02'
for filename in ['protocol.json','completion_status.json','progress.json']:
    path=run/filename
    if path.exists():
        print(filename);display(json.loads(path.read_text()))

protocol.json


{'version': 2,
 'config_sha256': '83ce9e0411f82c28a20914adc63958f5e36b80a1ce6871002fbfef0fafb6e5c4',
 'split_sha256': '262bb2a91c969b6b844842acffe90522ea98a5c2a0d892eebb0cf174efa9d06e',
 'counts': {'fit': 804,
  'tune': 226,
  'official_valid': 120,
  'calibration': 100},
 'training_data': 'MRNet only',
 'selection': 'Internal tuning macro AUROC; incumbent included; no external metric-based selection',
 'calibration': '100 cases removed from old fitting partition; candidate-only calibration after selection',
 'operating_point': 'Maximum Youden J on reserved calibration cohort; not a clinical operating point',
 'budget': 'Two architectures, one seed, up to 6 fine-tuning epochs each, patience 3',
 'limitations': ['Study independence only; patient linkage unavailable',
  'Internal tuning and official validation previously evaluated',
  'Incumbent previously trained on calibration subset: never calibrate incumbent on it',
  'New candidate uses 804 fitting cases vs incumbent 904; comparison

completion_status.json


{'stage': 'ready_for_research_testing',
 'updated_at': '2026-09-26T23:36:15.760828+00:00',
 'clinical_ready': False,
 'internal_auroc_improved': True,
 'overall_improvement_demonstrated': False}

progress.json


{'stage': 'fine_tuning',
 'architecture': 'efficientnet_b0',
 'epoch': 3,
 'studies': 800,
 'total': 804,
 'elapsed_seconds': 569}

In [2]:
for filename in ['results/model_comparison.json','results/calibration.json','validation_summary.json','verification.json']:
    path=run/filename
    if path.exists():
        print(filename);display(json.loads(path.read_text()))
    else:print(filename, 'not produced yet; no final result claimed')

results/model_comparison.json


{'criterion': 'Internal tuning macro AUROC only',
 'winner': 'efficientnet_b0',
 'candidate_improved': True,
 'models': [{'architecture': 'resnet18',
   'selected_epoch': 5,
   'stage': 'final_encoder_blocks_finetuned',
   'metrics': {'abnormal': {'auroc': 0.8725378065827932,
     'average_precision': 0.9680828384676412,
     'f1': 0.8536585365853658,
     'sensitivity': 0.7650273224043715,
     'specificity': 0.8837209302325582,
     'precision': 0.9655172413793104,
     'accuracy': 0.7876106194690266,
     'threshold': 0.5,
     'confusion_matrix': [[38, 5], [43, 140]],
     'positive_cases': 183,
     'negative_cases': 43},
    'acl': {'auroc': 0.8421325051759833,
     'average_precision': 0.6982010992482534,
     'f1': 0.5454545454545454,
     'sensitivity': 0.7142857142857143,
     'specificity': 0.7934782608695652,
     'precision': 0.4411764705882353,
     'accuracy': 0.7787610619469026,
     'threshold': 0.5,
     'confusion_matrix': [[146, 38], [12, 30]],
     'positive_cases'

results/calibration.json


{'calibration': {'method': 'positive_slope_platt',
  'source': 'reserved_MRNet_calibration',
  'cases': 100,
  'parameters': [{'scale': 1.0088440235212799, 'bias': 1.0287242305769888},
   {'scale': 0.7962809359337915, 'bias': -0.9477347946852002},
   {'scale': 0.933515933296675, 'bias': -0.3847124633408822}],
  'limitation': 'Small development calibration cohort; external clinical calibration is not established.'},
 'thresholds': {'abnormal': 0.6943677953637496,
  'acl': 0.24608272654141142,
  'meniscus': 0.4807751497984078},
 'before': {'abnormal': {'brier': 0.15290875732898712,
   'log_loss': 0.46432411670684814},
  'acl': {'brier': 0.15891219675540924, 'log_loss': 0.4825417399406433},
  'meniscus': {'brier': 0.15332479774951935, 'log_loss': 0.481648713350296}},
 'after': {'abnormal': {'brier': 0.12042995505653419,
   'log_loss': 0.37182843750233663},
  'acl': {'brier': 0.12588526557653906, 'log_loss': 0.40333158143684744},
  'meniscus': {'brier': 0.1484221585617518, 'log_loss': 0.46

validation_summary.json


{'candidate_improved_internal_auroc': True,
 'selected_architecture': 'efficientnet_b0',
 'selected_stage': 'frozen_warmup',
 'mrnet': {'dataset': 'MRNet-v1.0',
  'architecture': 'efficientnet_b0',
  'partition': 'official_valid',
  'exams': 120,
  'metrics': {'abnormal': {'auroc': 0.9006315789473684,
    'average_precision': 0.9679541342240485,
    'f1': 0.9270833333333334,
    'sensitivity': 0.9368421052631579,
    'specificity': 0.68,
    'precision': 0.9175257731958762,
    'accuracy': 0.8833333333333333,
    'threshold': 0.6943677953637496,
    'confusion_matrix': [[17, 8], [6, 89]],
    'positive_cases': 95,
    'negative_cases': 25},
   'acl': {'auroc': 0.8251964085297419,
    'average_precision': 0.7991313931018582,
    'f1': 0.7457627118644068,
    'sensitivity': 0.8148148148148148,
    'specificity': 0.696969696969697,
    'precision': 0.6875,
    'accuracy': 0.75,
    'threshold': 0.24608272654141142,
    'confusion_matrix': [[46, 20], [10, 44]],
    'positive_cases': 54,
  

verification.json


{'software_pipeline_passed': True,
 'browser_verification_pending': False,
 'clinical_ready': False,
 'selected_checkpoint': 'runs/mri_finetune_02/models/best_model.pth',
 'browser_verified_at': '2026-09-26T23:36:15.760828+00:00',
 'checkpoint_sha256': 'b60293f6c7130fd646c81b6d758584faf3e17b2689503b5278ad711156d546da',
 'tests_passed': 23,
 'notebooks_passed': 9,
 'real_upload': True,
 'predictions': True,
 'gradcam': True,
 'json_export': True,
 'clear_case': True,
 'empty_input_error': True,
 'picture_upload_removed': True}

Selection uses internal tuning macro AUROC only. Calibration and a balanced sensitivity/specificity threshold use the reserved calibration partition. External scores never trigger tuning or model selection. A candidate that does not beat the incumbent is retained as an experiment, and the incumbent remains selected. This experiment uses one seed and is not proof of clinical readiness.